
# Skills
Skills are reusable agent capabilities that provide specialized workflows and domain knowledge. 

- You can use Agent Skills to provide your deep agent with new capabilities and expertise. For ready-to-use skills that improve your agent’s performance on LangChain ecosystem tasks, see the LangChain Skills repository. 
- Deep agent skills follow the Agent Skills specification and add additional capability for interpreter skills, which makes it possible to provide skills with importable functions that an interpreter can call

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model

# Option 1: Fast, highly capable, and excellent with native tool-calling schemas
model = init_chat_model("google_genai:gemini-2.5-flash")
model 

ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), location=None, model='gemini-2.5-flash', client=<google.genai.client.Client object at 0x00000265D9A4CAD0>, default_metadata=(), model_kwargs={})

## Setup ollama (local models) - Independent from third party and their token limits

In [3]:
from langchain.chat_models import init_chat_model

# Initialize your local model via LangChain's uniform API
model = init_chat_model(
    model="llama3.1:8b",        # Replace with your local model tag (e.g., "qwen2.5:14b")
    model_provider="ollama",
    base_url="http://localhost:11434" # Default local Ollama endpoint
)


ImportError: Initializing ChatOllama requires the langchain-ollama package. Please install it with `pip install langchain-ollama`

In [ ]:
from deepagents import create_deep_agent
agent = create_deep_agent(
    model=model,
    system_prompt=(
        "You are a research assistant specializing in scientific literature. "
        "Always cite sources. Use subagents for parallel research on different topics."
    ),
)

agent

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is deepagent?"}]})
result

In [ ]:
from pathlib import Path
agents_md=Path("../projects/AGENTS.md").read_text(encoding="utf-8")
agents_md

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
checkpointer=MemorySaver()


## Skills
Skills are reusable agent capabilities that provide specialized workflows and domain knowledge. You can use Agent Skills to provide your deep agent with new capabilities and expertise. For ready-to-use skills that improve your agent’s performance on LangChain ecosystem tasks, see the LangChain Skills repository. Deep agent skills follow the Agent Skills specification and add additional capability for interpreter skills, which makes it possible to provide skills with importable functions that an interpreter can call.

In [ ]:

from urllib.request import urlopen
from deepagents import create_deep_agent
from deepagents.backends import StateBackend
from deepagents.backends.utils import create_file_data
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
backend = StateBackend()

from pathlib import Path
skills_content= Path("../skills/langgraph/SKILL.md").read_text(encoding="utf-8")

skills_files = {
    "../skills/langgraph/SKILL.md": create_file_data(skills_content),
}

In [ ]:

# Read every skills/<name>/SKILL.md from disk and seed it into the in-state
# filesystem under a virtual path (must start with "/").
skill_dirs = ["langgraph", "python", "aws", "report-writer"]
skills_files = {
    f"../skills/{name}/SKILL.md": create_file_data(
        Path(f"../skills/{name}/SKILL.md").read_text(encoding="utf-8")
    )
    for name in skill_dirs
}

In [ ]:
skills_files

In [ ]:
agent=create_deep_agent(
    model=model,
    backend=backend,
    skills=["../skills/"],
    checkpointer=checkpointer
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "What skills do you have available, and when would you use each?"}],
        
        "files": skills_files,
    },
    config={"configurable": {"thread_id": "skills-demo1"}},
)

print(result["messages"][-1].content)